# SE1_data_access

This notebook documents reproducible access to a subset of public raw datasets used in the HydroReg-Oulujoki project. Direct-download files are stored outside the repository, and one public FMI API request is preserved as a raw response file for transparency and reproducibility.

## Goals
- download public raw datasets from stable URLs
- store raw files outside the Git repository
- fetch and save Syke OData hydrology API responses (stations, discharge, water level, evaporation)
- document one public FMI WFS API request for reproducibility
- keep raw-data access separate from preprocessing and modelling

This notebook focuses on **data access only**.
It does not run the full hydrological or hydropower workflow.


In [1]:
from pathlib import Path
import requests
import os
import time
import json
import logging
from urllib.parse import urlparse
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

## Create Raw data folder

In [2]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "README.md").exists():
            return p
    return start

notebook_dir = Path.cwd()
repo_root = find_repo_root(notebook_dir)
external_base = repo_root.parent

raw_data_dir = external_base / "HydroReg-Oulujoki_rawdata"
direct_dir = raw_data_dir / "direct_downloads"
api_dir = raw_data_dir / "api"

direct_dir.mkdir(parents=True, exist_ok=True)
api_dir.mkdir(parents=True, exist_ok=True)

print("Notebook directory :", notebook_dir)
print("Repository root    :", repo_root)
print("Raw data directory :", raw_data_dir)

Notebook directory : /home/jovyan/WaterDig/ClassData
Repository root    : /home/jovyan/WaterDig
Raw data directory : /home/jovyan/HydroReg-Oulujoki_rawdata


## Data description

This notebook downloads the public raw datasets required for the HydroReg-Oulujoki workflow.

### Data groups
1. **Direct-download spatial datasets** (Syke open data)
   - catchments (`valumaalueet.zip`)
   - land cover CLC 2018 Finland 20 m (`clc2018_fi20m.zip`)
   - land cover CLC 2018 EU 25 ha (`clc2018eu25ha.zip`)

2. **Syke hydrology OData API** (Hydrologiarajapinta 1.1)
   - stations (`Paikka`) → `api/syke_Paikka.json`
   - discharge (`Virtaama`) → `api/syke_Virtaama.json`
   - water level (`Vedenkorkeus`) → `api/syke_Vedenkorkeus.json`
   - evaporation (`Haihdunta`) → `api/syke_Haihdunta.json`

3. **FMI WFS 2.0 API example**
   - hourly weather observations for Oulu → `api/fmi_oulu_*.xml`

### Purpose
- spatial datasets: basin/sub-catchment delineation and HRU preprocessing
- Syke hydrology: HBV calibration/validation targets; lake mass balance; operational envelope
- FMI observations: hydrological forcing preparation (P, T)

### Scope
This notebook performs **reproducible raw-data access only**.
It does not perform spatial preprocessing, HBV calibration, FlexTool scheduling, or scenario analysis.

## How to run this notebook

### Inputs
- stable public URLs from `datalinks.txt`
- one public FMI API request

### Outputs
Raw files are stored **outside the Git repository** in:

`../HydroReg-Oulujoki_rawdata/`

Subfolders:
- `direct_downloads/`
- `api/`

### Storage requirements
Storage demand is low to moderate for this notebook.
The largest files are the land-cover zip datasets.

In [10]:
!df -k ..

Filesystem     1K-blocks    Used Available Use% Mounted on
/dev/sdb        10218772 5765228   4437160  57% /home/jovyan


## Helper functions

In [3]:
def get_filename_from_url(url):
    path = urlparse(url).path
    name = os.path.basename(path)
    return name if name else "downloaded_file"

def download_file(url, output_dir, timeout=60):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = get_filename_from_url(url)
    filepath = output_dir / filename

    if filepath.exists():
        print(f"File already exists, skipping: {filename}")
        return filepath

    try:
        print(f"Downloading: {url}")
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        print(f"Saved: {filepath}")
        return filepath

    except Exception as e:
        print(f"Failed: {url}")
        print(f"Error: {e}")
        return None

## Links

In [4]:
possible_link_files = [
    repo_root / "inputs" / "datalinks.txt",
    repo_root / "datalinks.txt",
    notebook_dir / "datalinks.txt",
]

links_file = None
for p in possible_link_files:
    if p.exists():
        links_file = p
        break

if links_file is None:
    raise FileNotFoundError("Could not find datalinks.txt")

print("Using links file:", links_file)

with open(links_file, "r", encoding="utf-8") as f:
    all_links = [line.strip() for line in f if line.strip()]

print(f"Found {len(all_links)} links in datalinks.txt")
pd.DataFrame({"url": all_links})

Using links file: /home/jovyan/WaterDig/ClassData/datalinks.txt
Found 74 links in datalinks.txt


,url
0,# HydroReg-Oulujoki dataset links
1,# Concrete official URLs and example API queri...
2,# Replace dates / tokens / station IDs as needed.
3,# Do not store private tokens in this file.
4,# --------------------------------------------...
...,...
69,# --------------------------------------------...
70,# - ENTSO-E token must be kept in an environme...
71,# - SYKE/FMI queries can be narrowed by statio...
72,"# - For full reproducibility, create a second ..."


## Direct-download files vs API endpoints vs reference webpages

The links file contains mixed resource types:
- direct downloadable files (for example `.zip`)
- API/service endpoints
- reference webpages

For reproducible raw-data access, these resource types should be handled separately.
This notebook automatically downloads only the **direct file URLs** and stores the **FMI API response** separately.

In [5]:
direct_urls = [u for u in all_links if u.lower().endswith(".zip")]

print("Direct-download URLs:")
for u in direct_urls:
    print("-", u)

downloaded_direct = []
for url in direct_urls:
    fp = download_file(url, direct_dir)
    downloaded_direct.append(fp)
    time.sleep(2)

downloaded_direct

Direct-download URLs:
- SYKE_CATCHMENT_DIVISION_ZIP: https://wwwd3.ymparisto.fi/d3/gis_data/spesific/valumaalueet.zip
- SYKE_CORINE_2018_RASTER_20M_ZIP: https://wwwd3.ymparisto.fi/d3/Static_rs/spesific/clc2018_fi20m.zip
- SYKE_CORINE_2018_VECTOR_25HA_ZIP: https://wwwd3.ymparisto.fi/d3/Static_rs/spesific/clc2018eu25ha.zip
File already exists, skipping: valumaalueet.zip
File already exists, skipping: clc2018_fi20m.zip
File already exists, skipping: clc2018eu25ha.zip


[PosixPath('/home/jovyan/HydroReg-Oulujoki_rawdata/direct_downloads/valumaalueet.zip'),
 PosixPath('/home/jovyan/HydroReg-Oulujoki_rawdata/direct_downloads/clc2018_fi20m.zip'),
 PosixPath('/home/jovyan/HydroReg-Oulujoki_rawdata/direct_downloads/clc2018eu25ha.zip')]

## Syke hydrology OData API

Fetch and save the four public Syke hydrology OData endpoints required by `SE2_data_processing.ipynb`.
Each endpoint returns an OData JSON response; pagination is handled automatically.
Files are saved to `api/syke_<Endpoint>.json` exactly as expected by SE2.

**Base URL:** `https://rajapinnat.ymparisto.fi/api/Hydrologiarajapinta/1.1/odata/`  
**Authentication:** none required (public endpoint)  
**Note:** the `Virtaama`, `Vedenkorkeus`, and `Haihdunta` endpoints can return very large datasets.
Optionally add `$filter` query parameters below to restrict to Oulujoki basin stations and the study period (2014–2024).

In [14]:
# ── Configuration ────────────────────────────────────────────────────────────
STUDY_START         = '2014-01-01T00:00:00Z'
STUDY_END           = '2024-12-31T23:59:59Z'
BASIN_NAME_CONTAINS = 'Oulujoki'
BASIN_FIELD         = 'PaaVesalNimi'
STATION_ID_KEY      = 'Paikka_Id'
CHUNK_SIZE          = 15    # IDs per request — keep URL under ~2000 chars
SYKE_BASE           = 'https://rajapinnat.ymparisto.fi/api/Hydrologiarajapinta/1.1/odata'


# ── Helpers ──────────────────────────────────────────────────────────────────
def fetch_odata_all_pages(url: str, timeout: int = 120) -> list:
    records, next_url, page = [], url, 1
    while next_url:
        r = requests.get(next_url, timeout=timeout, headers={'Accept': 'application/json'})
        r.raise_for_status()
        data = r.json()
        records.extend(data.get('value', data))
        next_url = data.get('@odata.nextLink') or data.get('odata.nextLink')
        page += 1
        time.sleep(1)
    return records

def save_json(records: list, path: Path) -> str:
    path.write_text(json.dumps({'value': records}, ensure_ascii=False, indent=2))
    size_mb = round(path.stat().st_size / 1e6, 2)
    return f'{len(records)} records, {size_mb} MB'

def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def fetch_by_station_chunks(endpoint, station_ids, start, end, chunk_size=CHUNK_SIZE):
    all_records = []
    total_chunks = (len(station_ids) + chunk_size - 1) // chunk_size
    for i, chunk in enumerate(chunks(station_ids, chunk_size)):
        id_filter   = ' or '.join(f'Paikka_Id eq {sid}' for sid in chunk)
        time_filter = f"Aika ge datetime'{start}' and Aika le datetime'{end}'"
        url = f'{SYKE_BASE}/{endpoint}?$filter=({id_filter}) and {time_filter}'
        records = fetch_odata_all_pages(url)
        all_records.extend(records)
        print(f'\r  {endpoint}: chunk {i+1}/{total_chunks} — {len(all_records)} records so far', end='', flush=True)
        time.sleep(1)
    print()  # newline after progress line
    return all_records


# ── Step 1: load Paikka and resolve Oulujoki station IDs ─────────────────────
paikka_path = api_dir / 'syke_Paikka.json'

if paikka_path.exists():
    log.info('syke_Paikka.json already exists, loading from disk')
    all_stations = json.loads(paikka_path.read_text()).get('value', [])
else:
    log.info('Fetching Paikka (all stations)...')
    all_stations = fetch_odata_all_pages(f'{SYKE_BASE}/Paikka')
    save_json(all_stations, paikka_path)

oulujoki_stations = [
    s for s in all_stations
    if BASIN_NAME_CONTAINS.lower() in str(s.get(BASIN_FIELD, '')).lower()
]
station_ids = [s[STATION_ID_KEY] for s in oulujoki_stations if STATION_ID_KEY in s]

print(f'Total stations in Paikka : {len(all_stations)}')
print(f'Oulujoki stations found  : {len(oulujoki_stations)}')
print(f'Fetching in chunks of    : {CHUNK_SIZE} → {(len(station_ids) + CHUNK_SIZE - 1) // CHUNK_SIZE} requests per endpoint')


# ── Step 2: fetch time-series endpoints in chunks ────────────────────────────
TIME_SERIES_ENDPOINTS = {
    'syke_Virtaama.json'    : 'Virtaama',
    'syke_Vedenkorkeus.json': 'Vedenkorkeus',
    'syke_Haihdunta.json'   : 'Haihdunta',
}

syke_results = {'syke_Paikka.json': f'{len(oulujoki_stations)} Oulujoki stations'}

for filename, endpoint in TIME_SERIES_ENDPOINTS.items():
    out_path = api_dir / filename
    if out_path.exists():
        log.info(f'Already exists, skipping: {filename}')
        syke_results[filename] = 'skipped (already exists)'
        continue
    log.info(f'Fetching {filename} ({endpoint}) in chunks...')
    records = fetch_by_station_chunks(endpoint, station_ids, STUDY_START, STUDY_END)
    syke_results[filename] = save_json(records, out_path)
    log.info(f'Saved: {out_path} — {syke_results[filename]}')
    time.sleep(2)

print('\nSyke OData fetch summary:')
for fname, status in syke_results.items():
    print(f'  {fname:35s}: {status}')


2026-03-20 10:24:21,895 INFO syke_Paikka.json already exists, loading from disk
2026-03-20 10:24:21,930 INFO Fetching syke_Virtaama.json (Virtaama) in chunks...


Total stations in Paikka : 3966
Oulujoki stations found  : 249
Fetching in chunks of    : 15 → 17 requests per endpoint
  Virtaama: chunk 17/17 — 125786 records so far


2026-03-20 10:30:15,188 INFO Saved: /home/jovyan/HydroReg-Oulujoki_rawdata/api/syke_Virtaama.json — 125786 records, 24.13 MB
2026-03-20 10:30:17,190 INFO Fetching syke_Vedenkorkeus.json (Vedenkorkeus) in chunks...


  Vedenkorkeus: chunk 17/17 — 114059 records so far


2026-03-20 10:35:26,610 INFO Saved: /home/jovyan/HydroReg-Oulujoki_rawdata/api/syke_Vedenkorkeus.json — 114059 records, 18.66 MB
2026-03-20 10:35:28,611 INFO Fetching syke_Haihdunta.json (Haihdunta) in chunks...


  Haihdunta: chunk 17/17 — 1128 records so far

2026-03-20 10:36:05,861 INFO Saved: /home/jovyan/HydroReg-Oulujoki_rawdata/api/syke_Haihdunta.json — 1128 records, 0.13 MB




Syke OData fetch summary:
  syke_Paikka.json                   : 249 Oulujoki stations
  syke_Virtaama.json                 : 125786 records, 24.13 MB
  syke_Vedenkorkeus.json             : 114059 records, 18.66 MB
  syke_Haihdunta.json                : 1128 records, 0.13 MB


## FMI WFS 2.0 API example

Fetch a sample of hourly weather observations from the FMI Open Data WFS service and save the
raw XML response for reproducibility. Extend the date range and/or loop over additional
stored queries in `SE2_data_processing.ipynb` as needed for the full study period.

In [15]:
fmi_query = (
    "https://opendata.fmi.fi/wfs"
    "?service=WFS"
    "&version=2.0.0"
    "&request=getFeature"
    "&storedquery_id=fmi::observations::weather::hourly::timevaluepair"
    "&place=oulu"
    "&starttime=2024-01-01T00:00:00Z"
    "&endtime=2024-01-03T00:00:00Z"
    "&timestep=60"
)

response = requests.get(fmi_query, timeout=90)

print("Status:", response.status_code)
print(response.text[:1000])   

response.raise_for_status()

fmi_outfile = api_dir / "fmi_oulu_2024-01-01_2024-01-03_hourly.xml"
with open(fmi_outfile, "wb") as f:
    f.write(response.content)

print("Saved:", fmi_outfile)

Status: 200
<?xml version="1.0" encoding="UTF-8"?>
<wfs:FeatureCollection
    timeStamp="2026-03-20T10:39:36Z"
    numberMatched="12"
    numberReturned="12"
           xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
        xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:om="http://www.opengis.net/om/2.0"
        xmlns:ompr="http://inspire.ec.europa.eu/schemas/ompr/3.0"
        xmlns:omso="http://inspire.ec.europa.eu/schemas/omso/3.0"
        xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:gmd="http://www.isotc211.org/2005/gmd"
        xmlns:gco="http://www.isotc211.org/2005/gco" xmlns:swe="http://www.opengis.net/swe/2.0"
        xmlns:gmlcov="http://www.opengis.net/gmlcov/1.0"
        xmlns:sam="http://www.opengis.net/sampling/2.0"
        xmlns:sams="http://www.opengis.net/samplingSpatial/2.0"
        xmlns:wml2="http://www.opengis.net/waterml/2.0"
	xmlns:target="http://xml.fmi.fi/namespace/om/atmosphericfeatures/1.1"
        xsi:sc

## Notes on authentication and scope

- All direct-download datasets (Syke spatial) are public and require no authentication.
- All four Syke OData hydrology endpoints are public and require no authentication.
- The FMI WFS request used here is public and requires no authentication.
- ENTSO-E data are not downloaded in this notebook because token-based access must be documented separately.
- Fortum and Oulun Energia links are reference webpages and are not machine-readable raw datasets; plant characteristics are entered manually in SE2.
- If the Syke OData endpoints return very large responses (Virtaama / Vedenkorkeus), add `$filter` query parameters in the fetch cell above to restrict to Oulujoki basin stations and the 2014–2024 study period.

In [16]:
files = []
for p in raw_data_dir.rglob("*"):
    if p.is_file():
        files.append({
            "file": str(p),
            "size_mb": round(p.stat().st_size / 1024 / 1024, 2)
        })

files_df = pd.DataFrame(files).sort_values("file").reset_index(drop=True)
files_df

,file,size_mb
0,/home/jovyan/HydroReg-Oulujoki_rawdata/api/fmi...,0.22
1,/home/jovyan/HydroReg-Oulujoki_rawdata/api/syk...,0.12
2,/home/jovyan/HydroReg-Oulujoki_rawdata/api/syk...,2.84
3,/home/jovyan/HydroReg-Oulujoki_rawdata/api/syk...,17.80
4,/home/jovyan/HydroReg-Oulujoki_rawdata/api/syk...,23.01
5,/home/jovyan/HydroReg-Oulujoki_rawdata/direct_...,260.36
6,/home/jovyan/HydroReg-Oulujoki_rawdata/direct_...,109.16
7,/home/jovyan/HydroReg-Oulujoki_rawdata/direct_...,245.40


## Reproducibility summary

This notebook demonstrates a reproducible raw-data access pattern by:
1. reading documented public links,
2. separating file downloads from API requests,
3. storing raw outputs outside the repository,
4. preserving the exact API query used.

This supports transparent downstream preprocessing and modelling.

## Zenodo archiving note

For this project, a practical Zenodo strategy is:

- archive the repository snapshot
- archive processed/shareable outputs
- archive **lightweight provenance files** such as `download_manifest.jsonl`
- only publish raw data

## What you should have after running

After running all cells you should have the following in `../HydroReg-Oulujoki_rawdata/`:

**`direct_downloads/`**
- `valumaalueet.zip` — Syke sub-catchment polygons
- `clc2018_fi20m.zip` — CLC 2018 Finland 20 m raster
- `clc2018eu25ha.zip` — CLC 2018 EU 25 ha raster

**`api/`**
- `syke_Paikka.json` — Syke station metadata
- `syke_Virtaama.json` — Syke daily discharge
- `syke_Vedenkorkeus.json` — Syke daily water level
- `syke_Haihdunta.json` — Syke daily evaporation
- `fmi_oulu_2024-01-01_2024-01-03_hourly.xml` — FMI example WFS response

These files are the direct inputs to `SE2_data_processing.ipynb`.